# Notebook 03 — Huấn luyện mô hình (Model Training)
**DSP501 — Nhận diện người nói (Speaker Identification)**

## Mục tiêu
Huấn luyện **SVM (Support Vector Machine)** cho cả 2 pipeline và so sánh kết quả:

| Experiment | Pipeline | Features | Số chiều |
|------------|----------|----------|----------|
| **A1_SVM_basic** | A — Không DSP | RMS, ZCR, amplitude | 6 |
| **B1_SVM_dsp** | B — Có DSP | FIR + Pre-emphasis + MFCC | 26 |

### Chiến lược huấn luyện:
- **StratifiedKFold (5-fold)**: chia dữ liệu thành 5 phần, giữ tỷ lệ speakers đều
- **GridSearchCV**: thử 16 tổ hợp hyperparameters (C × gamma) để tìm bộ tốt nhất
- **StandardScaler trong Pipeline**: chuẩn hóa features → không bị data leakage
- **Random seed: 42**: đảm bảo kết quả reproducible

### Yêu cầu:
> Chạy **Notebook 02 (Feature Extraction)** trước để tạo file `.npy` trong `features/`

In [ ]:
import sys
sys.path.insert(0, '../src')

import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from train import train_svm, run_experiment, save_results
from evaluation import compute_ci, paired_ttest

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'

os.makedirs('../figures', exist_ok=True)
os.makedirs('../models', exist_ok=True)

print('Import thành công!')

---
## 1. Tải dữ liệu đặc trưng (Load Features)

Features đã được trích xuất ở Notebook 02 và lưu thành 3 file:

| File | Pipeline | Nội dung | Shape |
|------|----------|----------|-------|
| `features_basic.npy` | A | RMS, ZCR, amplitude | (100, 6) |
| `features_mfcc_filt.npy` | B | MFCC (mean + std) sau FIR + pre-emphasis | (100, 26) |
| `labels.npy` | — | Speaker IDs | (100,) |

In [ ]:
# Tải features
X_basic = np.load('../features/features_basic.npy')     # Pipeline A: 6 chiều
X_mfcc  = np.load('../features/features_mfcc_filt.npy')  # Pipeline B: 26 chiều
y       = np.load('../features/labels.npy')               # Nhãn speaker

# Thông tin speakers
speaker_names = {7: 'Cuong', 8: 'Quang', 9: 'Anne', 10: 'Khoa'}
labels_sorted = sorted(np.unique(y))
label_names = [speaker_names.get(i, f'Spk {i}') for i in labels_sorted]

print('=== DỮ LIỆU ĐẶC TRƯNG ===')
print(f'  Pipeline A (basic) : {X_basic.shape}  → {X_basic.shape[0]} mẫu × {X_basic.shape[1]} đặc trưng')
print(f'  Pipeline B (MFCC)  : {X_mfcc.shape}  → {X_mfcc.shape[0]} mẫu × {X_mfcc.shape[1]} đặc trưng')
print(f'  Labels             : {y.shape}')
print(f'  Speakers           : {label_names}')
print(f'  Số mẫu/speaker     : {dict(zip(label_names, [np.sum(y == i) for i in labels_sorted]))}')

# Kiểm tra dữ liệu
print(f'\n  Có NaN không?  basic={np.any(np.isnan(X_basic))}, mfcc={np.any(np.isnan(X_mfcc))}')
print(f'  Có Inf không?  basic={np.any(np.isinf(X_basic))}, mfcc={np.any(np.isinf(X_mfcc))}')

---
## 2. Giải thích SVM và quá trình huấn luyện

### SVM (Support Vector Machine) là gì?
- Thuật toán tìm **ranh giới tối ưu** (hyperplane) để phân chia các nhóm speakers
- Với kernel **RBF** (Radial Basis Function): ranh giới có thể **cong** → phù hợp dữ liệu phức tạp

### Hyperparameters cần tìm:

| Param | Ý nghĩa | Giá trị thử |
|-------|---------|------------|
| **C** | "Penalty" cho lỗi — C lớn → model cố phân loại đúng hết (dễ overfit) | 0.1, 1, 10, 100 |
| **gamma** | Phạm vi ảnh hưởng — gamma lớn → mỗi mẫu ảnh hưởng gần hơn (dễ overfit) | scale, auto, 0.001, 0.01 |

### Quy trình GridSearchCV:
```
Với mỗi tổ hợp (C, gamma):      ← 4 × 4 = 16 tổ hợp
  Với mỗi fold (1-3):            ← 3-fold CV bên trong
    Train trên 2 folds
    Test trên 1 fold
  Tính accuracy trung bình
Chọn tổ hợp có accuracy cao nhất
```

Tổng: 16 tổ hợp × 3 folds = **48 lần train** cho mỗi pipeline.

### StandardScaler trong Pipeline:
```python
Pipeline([
    ('scaler', StandardScaler()),  # Bước 1: chuẩn hóa → mean=0, std=1
    ('svm', SVC(kernel='rbf')),    # Bước 2: phân loại SVM
])
```
**Tại sao cần chuẩn hóa?** SVM nhạy cảm với **scale** của features. RMS Energy (~0.1) và ZCR (~0.05) có scale rất khác nhau → StandardScaler đưa về cùng scale.

**Tại sao đặt trong Pipeline?** Để scaler chỉ `fit` trên **training data** mỗi fold → **không bị data leakage** (không "nhìn trước" test data).

---
## 3. Huấn luyện Experiment A1 — Pipeline A (Không DSP)

Pipeline A dùng **6 đặc trưng cơ bản** miền thời gian:
- RMS Energy (mean, std)
- Zero Crossing Rate (mean, std) 
- Mean absolute amplitude
- Std amplitude

Đây là baseline — **không có bất kỳ kỹ thuật DSP nào** (không lọc, không FFT, không MFCC).

In [ ]:
print('Đang huấn luyện Pipeline A (basic features, không DSP)...')
print('  → GridSearchCV: thử 16 tổ hợp (C, gamma) × 3-fold = 48 lần train')
print()

res_a1, model_a1 = run_experiment('A1_SVM_basic', X_basic, y)

print(f'\n  Kết quả Pipeline A:')
print(f'  → Accuracy trung bình: {res_a1["accuracy"]["mean"]:.4f}')
print(f'  → 95% CI: [{res_a1["accuracy"]["ci_95"][0]:.4f}, {res_a1["accuracy"]["ci_95"][1]:.4f}]')
print(f'  → F1-macro: {res_a1["f1_macro"]["mean"]:.4f}')

---
## 4. Huấn luyện Experiment B1 — Pipeline B (Có DSP)

Pipeline B dùng **26 đặc trưng MFCC** sau khi xử lý DSP:
- Audio → FIR bandpass (300–3400 Hz) → Pre-emphasis (α=0.97) → MFCC (13 mean + 13 std)

**Kỳ vọng:** Pipeline B sẽ cho accuracy cao hơn Pipeline A vì:
1. FIR loại nhiễu → tín hiệu sạch hơn
2. Pre-emphasis → phổ phẳng hơn → MFCC chính xác hơn
3. MFCC capture **formants** (đặc trưng giọng nói) mà RMS/ZCR không thể

In [ ]:
print('Đang huấn luyện Pipeline B (MFCC sau DSP)...')
print('  → GridSearchCV: thử 16 tổ hợp (C, gamma) × 3-fold = 48 lần train')
print()

res_b1, model_b1 = run_experiment('B1_SVM_dsp', X_mfcc, y)

print(f'\n  Kết quả Pipeline B:')
print(f'  → Accuracy trung bình: {res_b1["accuracy"]["mean"]:.4f}')
print(f'  → 95% CI: [{res_b1["accuracy"]["ci_95"][0]:.4f}, {res_b1["accuracy"]["ci_95"][1]:.4f}]')
print(f'  → F1-macro: {res_b1["f1_macro"]["mean"]:.4f}')

---
## 5. So sánh CV Scores từng Fold

### Cross-Validation (5-fold) hoạt động thế nào?

Chia 100 mẫu thành 5 phần bằng nhau (mỗi phần 20 mẫu, 5 mẫu/speaker):

```
Fold 1: Train [2,3,4,5]  Test [1]  → Accuracy = ?
Fold 2: Train [1,3,4,5]  Test [2]  → Accuracy = ?
Fold 3: Train [1,2,4,5]  Test [3]  → Accuracy = ?
Fold 4: Train [1,2,3,5]  Test [4]  → Accuracy = ?
Fold 5: Train [1,2,3,4]  Test [5]  → Accuracy = ?
```

→ 5 accuracy scores → tính **mean ± std** và **95% CI**

**Ý nghĩa:**
- Scores **đều nhau** (std nhỏ) → model ổn định, generalize tốt
- Scores **dao động nhiều** (std lớn) → model không ổn định, phụ thuộc vào dữ liệu

**Ví dụ:** Pipeline A scores = [0.7, 1.0, 0.8, 0.9, 0.8] → std = 0.10 → dao động nhiều

In [ ]:
# Vẽ CV scores từng fold
folds = list(range(1, 6))
scores_a = res_a1['cv_scores']
scores_b = res_b1['cv_scores']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ đường
ax = axes[0]
ax.plot(folds, scores_a, 'o-', label=f'Pipeline A (mean={np.mean(scores_a):.3f})',
        color='steelblue', linewidth=2, markersize=8)
ax.plot(folds, scores_b, 's-', label=f'Pipeline B (mean={np.mean(scores_b):.3f})',
        color='darkorange', linewidth=2, markersize=8)
ax.set_xlabel('Fold')
ax.set_ylabel('Accuracy')
ax.set_title('CV Accuracy từng Fold')
ax.set_ylim(0, 1.1)
ax.set_xticks(folds)
ax.legend()
ax.grid(True, alpha=0.3)

# Thêm giá trị trên mỗi điểm
for i, (sa, sb) in enumerate(zip(scores_a, scores_b)):
    ax.annotate(f'{sa:.2f}', (folds[i], sa), textcoords='offset points',
               xytext=(0, 10), ha='center', fontsize=9, color='steelblue')
    ax.annotate(f'{sb:.2f}', (folds[i], sb), textcoords='offset points',
               xytext=(0, -15), ha='center', fontsize=9, color='darkorange')

# Biểu đồ cột so sánh mean
ax = axes[1]
x = np.arange(2)
means = [np.mean(scores_a), np.mean(scores_b)]
stds = [np.std(scores_a), np.std(scores_b)]
ci_a = compute_ci(np.array(scores_a))
ci_b = compute_ci(np.array(scores_b))
errors = [means[0] - ci_a[0], means[1] - ci_b[0]]

bars = ax.bar(x, means, yerr=errors, capsize=8,
              color=['steelblue', 'darkorange'], alpha=0.85, width=0.5)
ax.set_xticks(x)
ax.set_xticklabels(['Pipeline A\n(Không DSP)', 'Pipeline B\n(Có DSP)'])
ax.set_ylabel('Mean CV Accuracy')
ax.set_title('So sánh Accuracy với 95% CI')
ax.set_ylim(0, 1.15)
ax.grid(True, alpha=0.3, axis='y')

for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, m + errors[0] + 0.02,
            f'{m:.3f}\n(±{s:.3f})', ha='center', fontsize=10)

plt.suptitle('Cross-Validation Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/cv_scores.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nBảng tổng hợp:')
print(f'  Pipeline A: mean={np.mean(scores_a):.4f}, std={np.std(scores_a):.4f}, 95% CI=[{ci_a[0]:.4f}, {ci_a[1]:.4f}]')
print(f'  Pipeline B: mean={np.mean(scores_b):.4f}, std={np.std(scores_b):.4f}, 95% CI=[{ci_b[0]:.4f}, {ci_b[1]:.4f}]')

---
## 6. Kiểm định thống kê — Paired t-test

### Vấn đề:
Pipeline B có accuracy cao hơn, nhưng liệu sự khác biệt có **thật sự ý nghĩa** hay chỉ do **may mắn** ở cách chia dữ liệu?

### Paired t-test hoạt động thế nào?

1. Lấy 5 cặp scores: (A₁, B₁), (A₂, B₂), ..., (A₅, B₅)
2. Tính hiệu: dᵢ = Bᵢ - Aᵢ cho mỗi fold
3. Kiểm tra giả thuyết:
   - **H₀ (null)**: Không có khác biệt — mean(d) = 0
   - **H₁ (alternative)**: Có khác biệt — mean(d) ≠ 0

### Đọc kết quả:
- **p-value < 0.05** → Bác bỏ H₀ → Sự khác biệt **CÓ ý nghĩa thống kê** (confident > 95%)
- **p-value ≥ 0.05** → Không bác bỏ H₀ → **Chưa đủ bằng chứng** kết luận

### Ví dụ minh họa:
```
Fold 1: A=0.70, B=1.00 → d₁ = +0.30
Fold 2: A=1.00, B=1.00 → d₂ =  0.00
Fold 3: A=0.80, B=1.00 → d₃ = +0.20
Fold 4: A=0.90, B=1.00 → d₄ = +0.10
Fold 5: A=0.80, B=1.00 → d₅ = +0.20

mean(d) = 0.16 > 0 → B có vẻ tốt hơn
p-value = 0.035 < 0.05 → CÓ ý nghĩa thống kê!
```

In [ ]:
t_stat, p_value = paired_ttest(res_a1['cv_scores'], res_b1['cv_scores'])

# Tính hiệu từng fold
diffs = np.array(res_b1['cv_scores']) - np.array(res_a1['cv_scores'])

print('=' * 60)
print('   KIỂM ĐỊNH THỐNG KÊ: Paired t-test (A1 vs B1)')
print('=' * 60)
print()
print('  Hiệu (B - A) từng fold:')
for i, (a, b, d) in enumerate(zip(res_a1['cv_scores'], res_b1['cv_scores'], diffs)):
    print(f'    Fold {i+1}: A={a:.2f}, B={b:.2f} → d = {d:+.2f}')
print(f'\n  Mean(d)     = {diffs.mean():+.4f}')
print(f'  t-statistic = {t_stat:.4f}')
print(f'  p-value     = {p_value:.4f}')
print()
if p_value < 0.05:
    print(f'  >>> KẾT LUẬN: p = {p_value:.4f} < 0.05')
    print(f'  >>> DSP preprocessing CẢI THIỆN CÓ Ý NGHĨA thống kê!')
    print(f'  >>> Chỉ có {p_value*100:.1f}% khả năng kết quả này do ngẫu nhiên.')
else:
    print(f'  >>> KẾT LUẬN: p = {p_value:.4f} >= 0.05')
    print(f'  >>> Chưa đủ bằng chứng để kết luận DSP giúp cải thiện.')

---
## 7. Lưu mô hình & kết quả

Lưu lại để sử dụng trong:
- **Notebook 04** (Evaluation): đánh giá chi tiết trên test set
- **app.py** (Streamlit): demo nhận diện trực tiếp

### File được lưu:
| File | Nội dung |
|------|----------|
| `models/svm_pipeline_a.pkl` | Model Pipeline A (scaler + SVM) |
| `models/svm_pipeline_b.pkl` | Model Pipeline B (scaler + SVM) |
| `results.json` | Tất cả metrics, CV scores, hyperparameters, statistical tests |

In [ ]:
# Lưu models
joblib.dump(model_a1, '../models/svm_pipeline_a.pkl')
joblib.dump(model_b1, '../models/svm_pipeline_b.pkl')
print('Models đã lưu → models/svm_pipeline_a.pkl, svm_pipeline_b.pkl')

# Lưu kết quả
results = {
    'random_seed': 42,
    'cv_folds': 5,
    'experiments': {
        'A1_SVM_basic': res_a1,
        'B1_SVM_dsp':   res_b1,
    },
    'statistical_tests': {
        'SVM_A_vs_B': {'t_stat': t_stat, 'p_value': p_value}
    }
}
save_results(results, path='../results.json')
print('Kết quả đã lưu → results.json')

---
## 8. Tổng kết huấn luyện

In [ ]:
print('=' * 65)
print('                  TỔNG KẾT HUẤN LUYỆN')
print('=' * 65)
print(f'  Dataset: {len(y)} mẫu, {len(labels_sorted)} speakers ({', '.join(label_names)})')
print(f'  Phương pháp: SVM (RBF kernel) + GridSearchCV + 5-fold CV')
print()
print(f'  ┌──────────────────────────────────────────────────────────┐')
print(f'  │  Pipeline A (Không DSP — basic features):                │')
print(f'  │    Best params   : {res_a1["best_params"]}              │')
print(f'  │    CV Accuracy   : {res_a1["accuracy"]["mean"]:.4f} +/- {res_a1["accuracy"]["std"]:.4f}                    │')
print(f'  │    95% CI        : [{res_a1["accuracy"]["ci_95"][0]:.4f}, {res_a1["accuracy"]["ci_95"][1]:.4f}]              │')
print(f'  │    F1-macro      : {res_a1["f1_macro"]["mean"]:.4f}                              │')
print(f'  ├──────────────────────────────────────────────────────────┤')
print(f'  │  Pipeline B (Có DSP — FIR + Pre-emphasis + MFCC):       │')
print(f'  │    Best params   : {res_b1["best_params"]}              │')
print(f'  │    CV Accuracy   : {res_b1["accuracy"]["mean"]:.4f} +/- {res_b1["accuracy"]["std"]:.4f}                    │')
print(f'  │    95% CI        : [{res_b1["accuracy"]["ci_95"][0]:.4f}, {res_b1["accuracy"]["ci_95"][1]:.4f}]              │')
print(f'  │    F1-macro      : {res_b1["f1_macro"]["mean"]:.4f}                              │')
print(f'  ├──────────────────────────────────────────────────────────┤')
print(f'  │  Paired t-test:                                         │')
print(f'  │    t-statistic   : {t_stat:.4f}                              │')
print(f'  │    p-value       : {p_value:.4f}                               │')
print(f'  │    Kết luận      : {"CÓ" if p_value < 0.05 else "KHÔNG"} ý nghĩa thống kê (p {"<" if p_value < 0.05 else ">="} 0.05)    │')
print(f'  └──────────────────────────────────────────────────────────┘')
print()
print('  Tiếp theo: Notebook 04 — Đánh giá chi tiết trên Test Set')